## 1 · Imports

In [ ]:
from anndata import AnnData
from typing import Optional

# Libraries
import anndata as ad
import matplotlib as plt
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib.pyplot import rc_context
from scipy.stats import median_abs_deviation

from functools import partial
import altair as alt
import seaborn as sns
import decoupler as dc
from scipy.sparse import csr_matrix
import os
from pathlib import Path

import mudata as mu
import scanpy as sc
import scirpy as ir
import altair as alt
alt.data_transformers.enable("vegafusion")

import anndata as ad
import numpy as np
import palantir

import numpy as np
import scipy.sparse as sp

In [ ]:
import muon as mu
from muon import MuData

## 2 · Data Loading

Load mudata

In [ ]:
mdata = mu.read_h5mu("/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/40_gex_surface_prot/003_trajectory_annotated_mudata_normal.h5mu")

## 3 · Pseudobulk for DGSEA

Running it on mdata_without_cycling for DGSEA we want to avoid the cycling genes that could be misleading

In [ ]:
mdata["gex"].obs["all_cd8"] = "all"

In [ ]:
mdata.update()

In [ ]:
pdata = dc.pp.pseudobulk(
    adata=mdata["gex"],
    sample_col="sample_id",
    groups_col=["all_cd8"],   
    mode="sum",
    empty=False,
    layer="counts",
)

In [ ]:
pdata

In [ ]:
pdata.write_h5ad("pdata_normal.h5ad")

In [ ]:
sc.tl.pca(pdata)

In [ ]:
sc.pl.pca_variance_ratio(pdata)

In [ ]:
pdata

In [ ]:
sc.pl.pca(
    pdata,
    color=["sample_id",  "condition"],#,"cell_annotation_05"],
    ncols=3,
    size=300,
    frameon=True,
)

In [ ]:
# convert to array if sparse
counts = pdata.X

if not isinstance(counts, np.ndarray):
    counts = counts.toarray()

# keep genes with >=10 counts in at least 2 samples
gene_filter = (counts >= 10).sum(axis=0) >= 2

pdata_filtered = pdata[:, gene_filter].copy()

print("Genes before filtering:", pdata.shape[1])
print("Genes after filtering:", pdata_filtered.shape[1])

In [ ]:
from scipy import sparse

In [ ]:
samplesheet = pdata_filtered.obs.copy()
samplesheet.insert(0, "sample", pdata_filtered.obs_names)

samplesheet.to_csv("samplesheet_normal.csv", index=False)

X = pdata_filtered.X

if sparse.issparse(X):
    counts_array = X.toarray()
else:
    counts_array = np.asarray(X)

# genes as rows, samples as columns
counts_df = pd.DataFrame(
    counts_array.T,
    index=pdata_filtered.var_names,   # gene_name
    columns=pdata_filtered.obs_names
)


gene_id = pdata_filtered.var["gene_ids"].values      # ENSG...
gene_name = pdata_filtered.var_names.values          # gene symbols

# insert in correct order
counts_df.insert(0, "gene_name", gene_name)
counts_df.insert(0, "gene_id", gene_id)

# optional: ensure integer counts
counts_df.iloc[:, 2:] = counts_df.iloc[:, 2:].round().astype(int)

counts_df.columns = [
    col.replace("_all", "") if col not in ["gene_id", "gene_name"] else col
    for col in counts_df.columns
]

# save
counts_df.to_csv("counts_matrix_normal.tsv", sep="\t", index=False)

## 4 · Pseudobulk for cytosig

In [ ]:


pdata = dc.pp.pseudobulk(
    adata=mdata["gex"],
    sample_col="sample_id",
    groups_col=["all_cd8"],   
    mode="sum",
    empty=False,
    layer="counts",
)

In [ ]:
pdata.write_h5ad("pdata_cytosig_normal.h5ad")

## 5 · Pseudobulk for by cell type

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import sparse
import decoupler as dc
import scanpy as sc

cell_types = ["Naive",
    "Activated",
    "Tissue resident memory",
    "Cytotoxic/Effector memory",
]

outdir = "/data/scratch/kvalem/projects/2021/honda_microbial_metabolites_2021/20_scripts/40_single-cell-sorted-cd8/40_gex_surface_prot/28012026/run_gsea/tables/cell_type"
os.makedirs(outdir, exist_ok=True)

adata = mdata["gex"]

for cell_type in cell_types:

    print(f"Processing {cell_type}...")

    # -----------------------------
    # 1. Subset one cell type
    # -----------------------------
    adata_ct = adata[
        adata.obs["cell_annotation_05"] == cell_type
    ].copy()

    # -----------------------------
    # 2. Pseudobulk by sample
    # -----------------------------
    pdata = dc.pp.pseudobulk(
        adata=adata_ct,
        sample_col="sample_id",
        groups_col="cell_annotation_05",
        mode="sum",
        empty=False,
        layer="counts",
    )

    # Optional PCA QC
    sc.tl.pca(pdata)

    # -----------------------------
    # 3. Create samplesheet
    # -----------------------------
    samplesheet = pdata.obs.copy()
    samplesheet.insert(0, "sample", pdata.obs_names)

    safe_cell_type = cell_type.replace("/", "_")

    samplesheet.to_csv(
        f"{outdir}/samplesheet_{safe_cell_type}_normal.csv",
        index=False
    )

    # -----------------------------
    # 4. Create counts matrix
    # -----------------------------
    X = pdata.X

    if sparse.issparse(X):
        counts_array = X.toarray()
    else:
        counts_array = np.asarray(X)

    counts_df = pd.DataFrame(
        counts_array.T,
        index=pdata.var_names,
        columns=pdata.obs_names
    )
    # Remove the cell-type suffix from sample columns
    counts_df.columns = [
    col.removesuffix(f"_{cell_type}")
    if col not in ["gene_id", "gene_name"]
    else col
    for col in counts_df.columns
    ]

    gene_id = pdata.var["gene_ids"].values
    gene_name = pdata.var_names.values

    counts_df.insert(0, "gene_name", gene_name)
    counts_df.insert(0, "gene_id", gene_id)

    # Ensure integer counts
    counts_df.iloc[:, 2:] = counts_df.iloc[:, 2:].round().astype(int)

    # Optional: clean column names
    counts_df.columns = [
        col.replace("_all", "") if col not in ["gene_id", "gene_name"] else col
        for col in counts_df.columns
    ]

    counts_df.to_csv(
        f"{outdir}/counts_matrix_{safe_cell_type}_normal.tsv",
        sep="\t",
        index=False
    )